# Football Betting Models

Estimate fair decimal odds for home win, draw, and away win (1X2).

## Match data

Place `football_modelling.xlsx` in the working directory. Required columns are `kickoff`, both team IDs, and both team scores. The code uses NumPy, pandas, SciPy, scikit-learn, and openpyxl.

Hold out the latest ~20% of matches, keeping shared kickoff times together.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

columns = ["kickoff", "home_team_id", "away_team_id", "home_team_score", "away_team_score"]
matches = pd.read_excel("football_modelling.xlsx", usecols=columns)
matches["kickoff"] = pd.to_datetime(matches["kickoff"], errors="coerce")
matches = matches.dropna(subset=columns).sort_values("kickoff").reset_index(drop=True)

scores = matches[["home_team_score", "away_team_score"]].to_numpy(dtype=float)
if not np.all(np.isfinite(scores) & (scores >= 0) & (scores == np.floor(scores))):
    raise ValueError("Match scores must be non-negative whole numbers.")
if matches["kickoff"].nunique() < 2:
    raise ValueError("At least two distinct kickoff times are needed.")

cutoff = matches.loc[max(1, int(0.8 * len(matches))), "kickoff"]
if cutoff == matches.loc[0, "kickoff"]:
    cutoff = matches["kickoff"].drop_duplicates().iloc[1]
train = matches.loc[matches["kickoff"] < cutoff]
test = matches.loc[matches["kickoff"] >= cutoff]
print(f"Training matches: {len(train):,} | Test matches: {len(test):,} | Test starts: {cutoff}")

## Expected goals

Compare league-average scoring rates with a Poisson model that accounts for team attack, opponent defence, and home advantage.

In [ ]:
def scoring_rows(fixtures, is_home):
    attack, defence = ("home", "away") if is_home else ("away", "home")
    return pd.DataFrame({
        "is_home": int(is_home),
        "attack_team": fixtures[f"{attack}_team_id"].astype(str),
        "defence_team": fixtures[f"{defence}_team_id"].astype(str),
    })


training_features = pd.concat([scoring_rows(train, True), scoring_rows(train, False)])
training_goals = np.concatenate([train["home_team_score"], train["away_team_score"]])

features = ColumnTransformer([
    ("venue", "passthrough", ["is_home"]),
    ("teams", OneHotEncoder(handle_unknown="ignore", drop="first"),
     ["attack_team", "defence_team"]),
], sparse_threshold=1.0)
model = make_pipeline(features, PoissonRegressor(alpha=1e-6, max_iter=1000))
model.fit(training_features, training_goals)

score_columns = ["home_team_score", "away_team_score"]
baseline_rates = np.broadcast_to(train[score_columns].mean().to_numpy(), (len(test), 2))
team_rates = np.column_stack([
    model.predict(scoring_rows(test, True)),
    model.predict(scoring_rows(test, False)),
])

## Fair 1X2 odds

Combine independent home and away goal probabilities to price each outcome. Fair decimal odds equal 1 divided by the outcome probability.

In [ ]:
outcomes = ["home", "draw", "away"]


def outcome_probabilities(rates):
    home_rate, away_rate = rates.T
    max_home_goals = int(poisson.isf(1e-12, home_rate.max()))
    home_goals = np.arange(max_home_goals + 1)
    home_mass = poisson.pmf(home_goals, home_rate[:, None])
    home_win = np.sum(home_mass * poisson.cdf(home_goals - 1, away_rate[:, None]), axis=1)
    draw = np.sum(home_mass * poisson.pmf(home_goals, away_rate[:, None]), axis=1)
    away_win = np.sum(home_mass * poisson.sf(home_goals, away_rate[:, None]), axis=1)
    probabilities = np.column_stack([home_win, draw, away_win])
    return probabilities / probabilities.sum(axis=1, keepdims=True)


baseline_probabilities = outcome_probabilities(baseline_rates)
team_probabilities = outcome_probabilities(team_rates)

prices = test[["kickoff", "home_team_id", "away_team_id"]].copy()
for index, outcome in enumerate(outcomes):
    probability = team_probabilities[:, index]
    prices[f"{outcome}_probability"] = probability
    prices[f"{outcome}_fair_odds"] = np.divide(
        1, probability, out=np.full_like(probability, np.inf), where=probability > 0
    )
prices.head().round(4)

## Holdout evaluation

Compare scoreline log loss, 1X2 log loss, and 1X2 Brier score on the same fixtures. Lower scores are better.

In [ ]:
observed_scores = test[score_columns].to_numpy()
home_goals, away_goals = observed_scores.T
actual_outcomes = np.select([home_goals > away_goals, home_goals == away_goals], [0, 1], default=2)
actual_indicators = np.eye(len(outcomes))[actual_outcomes]


def evaluate(rates, probabilities):
    realized_probabilities = probabilities[np.arange(len(test)), actual_outcomes]
    return {
        "Scoreline log loss": -poisson.logpmf(observed_scores, rates).sum(axis=1).mean(),
        "1X2 log loss": -np.log(np.clip(realized_probabilities, 1e-15, 1)).mean(),
        "1X2 Brier score": np.square(probabilities - actual_indicators).sum(axis=1).mean(),
    }


pd.DataFrame({
    "League baseline": evaluate(baseline_rates, baseline_probabilities),
    "Team Poisson model": evaluate(team_rates, team_probabilities),
}).T.round(4)

## Pricing limitations

Independent scoring can misprice low-scoring draws. Testing betting returns requires timestamped bookmaker odds.